# 03 — Supervised GNN + Explainability

Step 4 (training) + Step 5/5.5 (explainability) in one notebook.

**What we predict.** We use the binary `sonicated` annotation as a demo
target. The GINE classifier (with adversarial debiasing against patient ID)
is trained with 5-fold patient-level CV on the niches.

**What we explain.** After training we run GNNExplainer + Integrated Gradients,
and turn the per-niche attributions into class-level summaries: top genes,
edge-feature channels, cell-type importance, LR-pair × cell-type-pair
interactions, pathway enrichment per cell type, and a niche-embedding UMAP.

Because the full run can take 15–20 min, this notebook drives the **prebuilt
pipelines** rather than redoing everything cell-by-cell.

In [ ]:
import sys; from pathlib import Path
REPO = Path.cwd().parent.parent
if str(REPO / 'src') not in sys.path: sys.path.insert(0, str(REPO / 'src'))

from ecofoundation.config.loader import load_config
from ecofoundation.pipelines.train import run_training_pipeline
from ecofoundation.pipelines.explain import (
    ExplainPipelineInputs, run_explainability_pipeline,
)

## 1. Train

Loads `configs/train_sonicated.yaml` (adversarial debiasing enabled by
default). Set `cfg.training.adversarial.enabled = False` if you want to
compare against the baseline.

In [ ]:
cfg = load_config(REPO / 'configs/train_sonicated.yaml')
# Uncomment to disable adversarial debiasing:
# cfg.training.adversarial.enabled = False
train_folder = run_training_pipeline(cfg)
print('Trained run:', train_folder.run_id)
print('Report     :', train_folder.report_path)

## 2. Explain

Loads the model checkpoint from the training run and produces the
explainability report (gene importance, LR × cell-type, pathway, embedding UMAP).

In [ ]:
cfg.run_name = 'explain_sonicated_nb'
inputs = ExplainPipelineInputs(
    source_model_path=train_folder.artifacts_dir / 'model_fold0.pt',
    source_predictions_path=train_folder.artifacts_dir / 'test_predictions.parquet',
    target_name='sonicated',
    top_k_per_outcome=8,
    gnn_explainer_epochs=80,
    ig_steps=16,
)
explain_folder = run_explainability_pipeline(cfg, inputs)
print('Explainability report:', explain_folder.report_path)

## 3. Inspect the artefacts

All outputs are persisted as parquet in `runs/<run_id>/artifacts/` and
as standalone PDFs in `runs/<run_id>/pdf/`.

In [ ]:
import pandas as pd
gene_imp_c0 = pd.read_parquet(explain_folder.artifacts_dir / 'gene_importance_class0.parquet')
gene_imp_c1 = pd.read_parquet(explain_folder.artifacts_dir / 'gene_importance_class1.parquet')
ct_imp     = pd.read_parquet(explain_folder.artifacts_dir / 'cell_type_importance.parquet')
lr_imp     = pd.read_parquet(explain_folder.artifacts_dir / 'lr_interaction_attribution.parquet')
pathways   = pd.read_parquet(explain_folder.artifacts_dir / 'pathway_enrichment.parquet')
gene_imp_c0.head(10)

## 4. Render plots individually

Every plot in the report has a corresponding factory function in
`ecofoundation.reporting.plots`. You can rebuild them with custom
parameters (top_n, colourmap, ...).

In [ ]:
from ecofoundation.reporting.plots import (
    top_gene_importance_bar, edge_channel_importance_bar,
    cell_type_importance_heatmap, lr_celltype_pair_heatmap,
    pathway_dotplot,
)
from ecofoundation.interpretation.aggregation import ClassAttribution

attr_c1 = ClassAttribution(
    target_class=1,
    n_niches_explained=int(gene_imp_c1['mean_abs_attr'].notna().sum()),
    gene_importance=gene_imp_c1,
    edge_channel_importance=pd.DataFrame(columns=['channel','mean_abs_attr']),
    node_mask_summary=pd.DataFrame(columns=['gene','mean_node_mask']),
)
fig = top_gene_importance_bar(attr_c1, top_n=15, class_label='Sonicated')
fig